In [ ]:
# A boundary case for variants V and V' is a trace with label V, but a length that is close (or equal) to the mean length of V'."

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pm4py
import random
random.seed(0)#set seed for reproducibility

In [2]:
def load_data(route_dataset, route_log):
    log=pm4py.read_xes(route_log)
    dataset=pd.read_csv(route_dataset,index_col=0)
    dataset = dataset.set_index("case:concept:name")
    X=dataset.drop(columns=["Class"])
    y=dataset['Class']

    le = LabelEncoder()
    print("Is na? "+str(X.isnull().values.any()))
    y_transformed = le.fit_transform(y)
    le_name_mapping = dict(zip(le.classes_,le.transform(le.classes_)))
    X_train_and_val, X_test, y_train_and_val, y_test = train_test_split(X,
                                                                        y_transformed,
                                                                        test_size=0.2,
                                                                        stratify=y_transformed,
                                                                        shuffle=True,#disorder the data
                                                                        random_state=0)

    X_train, X_val, y_train, y_val = train_test_split(X_train_and_val,
                                                      y_train_and_val,
                                                      test_size=0.2,
                                                      stratify=y_train_and_val,
                                                      shuffle=True,#disorder the data
                                                      random_state=0)
    
    return X_train, y_train, log, le_name_mapping

In [ ]:
def return_most_similar_cases(sizes_process_variant_v, mean_size_process_variant_v_prime):
    """
    Input:
        - sizes_process_variant: pandas series where each value correspond to the size of a case (i.e., total number of activites in a case) in a process variant v, 
          and the index corresponds to the case ids
        - mean_size_process_variant_v_prime: average size of the cases of process variant v'
    
    Output:
        - selected_case_id: return the case id belonging to variant v whose case size is most similar with the mean case size of variant v'

    """

    similarity_sizes_process_v_with_avg_v_prime=sizes_process_variant_v-mean_size_process_variant_v_prime#calculate how much the size of each case of process variant differs from the mean case size of process variant '
    similarity_sizes_v_with_avg_v_prime_abs=abs(similarity_sizes_process_v_with_avg_v_prime)#calculate how much they differ independently if they are bigger (positive difference) or shorter (negative difference)
    boundary_cases_candidates_of_v=similarity_sizes_v_with_avg_v_prime_abs[similarity_sizes_v_with_avg_v_prime_abs==min(similarity_sizes_v_with_avg_v_prime_abs)]#find the case or cases with minimum difference
    df_boundary_cases_process_variant_v=boundary_cases_candidates_of_v.reset_index()#reset the series index and convert it into a dataframe like this:
    #   case:concept:name	0
    #0         AA           20.4
    #1         CBA          0.7
    #...       XA           3.4

    if len(boundary_cases_candidates_of_v)>1:#if the size of the dataframe is bigger than one, it means that there is more than one case equal to the minimum case size
        #generate a random number to select a case of the dataframe. The random number will be the selected index
        index_selected_case = random.randint(0,len(df_boundary_cases_process_variant_v)-1)#the size of the pandas series is always one less, because it starts in zero
        selected_case_id=df_boundary_cases_process_variant_v.iloc[index_selected_case]["case:concept:name"]#get the selected case based on the random number
    else:#if there is only one case pick the only one
        selected_case_id=df_boundary_cases_process_variant_v.iloc[0]["case:concept:name"]

    return selected_case_id

def return_boundary_cases(log, X_train_dataset, y_train_dataset, selected_boundaries, le_name_mapping_series):

    """
    Input:
    - log: original log containing all the information of each case of each process variant
    - X_train_dataset: X matrix whose indices correspond to the cases
    - y_train_dataset: array containing the process variant of each case in X_train_dataset in numeric format (e.g.,[0,1,2,4,1,1,...])
    - selected_boundaryies: list with tuples containing the process variants where the boundary cases should be search ([ (Variant A, Variant B), (Variant B, Variant C), ... ])
    . le_name_mapping_series: pandas series whose indices are the real names of process variants of the process variant [Variant A: 0, Variant B:1,...]
    

    Output:
    - boundary_cases: dictionary whose keys represent the process variant boundaries, and the values are dictionary with the cases of each process variant related to the boundary 
    For example:
    {Variant A-Variant B: 
                {
                    Variant A: ["A", "B",...], 
                    Variant B:["A","C",...]
                },

    Variant B-Variant C:                {
                    Variant B: ["B", "B",...], 
                    Variant C:["D","C",...]
                },
                
                
    }
    """

    sizes_cases_variants={}#dictionary to store the case sizes of each process variant
    boundary_cases={}#output dictionary
    
    for label in set(y_train_dataset):#for each process variant in numeric format
        cases_variant_label=list(X_train_dataset[y_train_dataset==label].index)#get its case ids from the training dataset
        log_train_cases_variant_label=log[log["case:concept:name"].isin(cases_variant_label)]#get the rows (i.e. events) of each selected case id
        sizes_cases_variant_label=log_train_cases_variant_label.groupby("case:concept:name").apply(lambda x: len(x["concept:name"]))#group each case to its events and count how many activities it has (size)
        variant_name=le_name_mapping_series[le_name_mapping_series==label].index[0]#get the process variant name based on the numeric format
        sizes_cases_variants[variant_name]=sizes_cases_variant_label#store the case sizes relating them to the real name

    for boundaryFrontier in selected_boundaries:#for each boundary
        #get the two names of the process variants of the boundary
        boundaryX=boundaryFrontier[0]
        boundaryY=boundaryFrontier[1]

        #get their case sizes, adn calculate their mean case sizes
        sizes_cases_variantX=sizes_cases_variants[boundaryX]
        sizes_cases_variantY=sizes_cases_variants[boundaryY]
        mean_size_boundaryX=np.mean(sizes_cases_variantX)
        mean_size_boundaryY=np.mean(sizes_cases_variantY)

        #call this function to find the boundary case id of process variant X and process variant Y
        boundaryCaseX_case_id=return_most_similar_cases(sizes_process_variant_v=sizes_cases_variantX,
                                                        mean_size_process_variant_v_prime=mean_size_boundaryY)
        
        boundaryCaseY_case_id=return_most_similar_cases(sizes_process_variant_v=sizes_cases_variantY,
                                                        mean_size_process_variant_v_prime=mean_size_boundaryX)
        
        #get their activity sequences
        boundaryCaseX=log[log["case:concept:name"]==boundaryCaseX_case_id]["concept:name"].tolist()
        boundaryCaseY=log[log["case:concept:name"]==boundaryCaseY_case_id]["concept:name"].tolist()
        #create a dictionary to store their activity sequnences
        boundary={}
        boundary[boundaryX]=boundaryCaseX
        boundary[boundaryY]=boundaryCaseY
        #save it in the general dictionary
        boundary_cases[boundaryX+"-"+boundaryY]=boundary

    return boundary_cases

In [4]:
###########################################################################################################################################################################################################

In [27]:
#Sepsis log

In [5]:
X_train_sepsis, y_train_sepsis, sepsis_log, le_name_mapping_sepsis=load_data(route_dataset="./Data/sepsis/mined_sepsis_confidences_SIRS2OrMore.csv",
                                                                             route_log="./Data/sepsis/sepsis.xes")

c:\Users\ccagu\anaconda3\envs\naturalexamples\lib\site-packages\pm4py\util\dt_parsing\parser.py:77: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(
c:\Users\ccagu\anaconda3\envs\naturalexamples\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
parsing log, completed traces :: 100%|██████████| 1050/1050 [00:01<00:00, 926.16it/s]


Is na? True


In [6]:
le_name_mapping_sepsis

{'SIRS-False': 0, 'SIRS-True': 1}

In [7]:
alternative_bc_sepsis=return_boundary_cases(log=sepsis_log,
                                            X_train_dataset=X_train_sepsis,
                                            y_train_dataset=y_train_sepsis,
                                            selected_boundaries=[('SIRS-False','SIRS-True')],
                                            le_name_mapping_series=pd.Series(le_name_mapping_sepsis))

In [8]:
alternative_bc_sepsis["SIRS-False-SIRS-True"]["SIRS-False"]

['ER Registration',
 'ER Triage',
 'CRP',
 'LacticAcid',
 'Leucocytes',
 'ER Sepsis Triage',
 'Admission IC',
 'LacticAcid',
 'Leucocytes',
 'CRP',
 'LacticAcid',
 'Leucocytes',
 'CRP',
 'LacticAcid',
 'Admission NC',
 'Release A',
 'Return ER']

In [9]:
alternative_bc_sepsis["SIRS-False-SIRS-True"]["SIRS-True"]

['ER Registration',
 'ER Triage',
 'ER Sepsis Triage',
 'Leucocytes',
 'LacticAcid',
 'CRP',
 'IV Liquid',
 'IV Antibiotics']

In [22]:
#############################################################################################################################################################################################################

In [34]:
#RTFM

In [35]:
X_train_rtfm, y_train_rtfm, rtfm_log, le_name_mapping_rtfm=load_data(route_dataset="./Data/road_traffic/mined_rtfm_relabelled_confidences.csv",
                                                                     route_log="./Data/road_traffic/Road_Traffic_Fine_Management_Process.xes")

parsing log, completed traces :: 100%|██████████| 150370/150370 [00:31<00:00, 4770.78it/s]


Is na? True


In [36]:
le_name_mapping_rtfm

{'collected': 0, 'dismissed': 1, 'fully_paid': 2, 'unresolved': 3}

In [ ]:
alternative_bc_rtfm=return_boundary_cases(log=rtfm_log,
                                          X_train_dataset=X_train_rtfm,
                                          y_train_dataset=y_train_rtfm,
                                          selected_boundaries=[('dismissed','unresolved'),('fully_paid','unresolved')],
                                          le_name_mapping_series=pd.Series(le_name_mapping_rtfm))

In [39]:
alternative_bc_rtfm['dismissed-unresolved']

{'dismissed': ['Create Fine', 'Send Fine', 'Appeal to Judge'],
 'unresolved': ['Create Fine',
  'Send Fine',
  'Insert Fine Notification',
  'Insert Date Appeal to Prefecture',
  'Add penalty',
  'Send Appeal to Prefecture']}

In [41]:
alternative_bc_rtfm['fully_paid-unresolved']

{'fully_paid': ['Create Fine', 'Send Fine', 'Payment'],
 'unresolved': ['Create Fine', 'Payment', 'Send Fine']}